# Here bagging is manually coded wihout any use of sklearn library, core idea how it works is shown

# Bagging 
- Bagging means Bootstrap Aggregation. 

In statistics, Bootstrapping usually means drawing a sample randomly from a dataset. 

same concept is used by bagging, we know bagging is an type of Ensemble Learning so obviously it will have an collective model called ensemble and many basline models under the ensemble. 


what we do is, say we have 10k dataset and we want to prvide 1k dataset to each model then we randomly draw the sample for 10k data set and provide to each model .. 
- it can be done with replacement( after drawing D1-1k row for model 1 we put back the data to 10k dataset and again randomly select as D2 for model2 )
- or we can do wihout replacement( not putting back the dataset used for one model )


we are randomly selecting data/ rows so it auto creates variation most model have very high probabilty that it will trainin diffrent 1k dataset form 10k dataset..

Now after giving each model say we took 3 model then we train with random 1k D1,d2,d3 dataset . now we have a new query point for which we have to make predection (WE ARE TALKING ABOUT CLSSIFICATION HERE)   then we provide the query point to each model and generate the ouput and do the majority count to find the final output for the new wuery point . 


upto training process is called bootstrappig and after it genertion prdeciton, majority count is called aggregation . 

> what can we do with Bagging then ? 

you should know about bias and variance tradeoff 

bias means the model cant even capture the patterns in training data. 

while variance measn if theres is slight change in data the model will provide diffrent output so the output varies .. 

you dont get low bias and low variance model so bias variance trade-off says we should go for model which come near them ... 


now These models like D.T, svm , knn overfit(they pefrom extremly well in traning data but not in testing ) and what is bias we leant above so it is a low bias model . >>>> Decision Trees already have low bias, but they suffer from high variance.

you eithr get low bias high variance model or high bias low variance mdoel 


but with the help of bagging we can avhive the lb,lv model 

it has little to no effect on bias so we aremainly thinknig for reducing variance 

> How does Bagging reduce variance?

Suppose each model is trained on 1,000 rows. We are not manually changing 100 rows. Instead, because we create each dataset using random sampling with replacement, each model naturally receives a slightly different dataset.
For example:
Model 1 may differ from Model 2 by around 100 rows.
Model 2 may differ from Model 3 by around 50 rows.
Some rows may even appear multiple times in one dataset, while some may not appear at all.
Since each model sees a slightly different version of the data, each learns slightly different patterns and makes different mistakes. When we combine all their predictions using majority voting, these individual mistakes tend to cancel out, reducing variance while keeping the bias nearly the same.
This preserves the intuition you were aimi

In [637]:
import pandas as pd 
import numpy as np 
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import accuracy_score



In [605]:
# first we will see how will decisoin tree peform individually, 

wine=load_wine()
data=pd.DataFrame(wine.data, columns=wine.feature_names)
data['target']=wine.target

In [606]:
data

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


In [607]:
data['target']

0      0
1      0
2      0
3      0
4      0
      ..
173    2
174    2
175    2
176    2
177    2
Name: target, Length: 178, dtype: int64

target is like serial so lests suffle this data 

In [608]:
data.shape

(178, 14)

In [609]:
df=data.sample(178)
df

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
106,12.25,1.73,2.12,19.0,80.0,1.65,2.03,0.37,1.63,3.40,1.00,3.17,510.0,1
120,11.45,2.40,2.42,20.0,96.0,2.90,2.79,0.32,1.83,3.25,0.80,3.39,625.0,1
6,14.39,1.87,2.45,14.6,96.0,2.50,2.52,0.30,1.98,5.25,1.02,3.58,1290.0,0
90,12.08,1.83,2.32,18.5,81.0,1.60,1.50,0.52,1.64,2.40,1.08,2.27,480.0,1
172,14.16,2.51,2.48,20.0,91.0,1.68,0.70,0.44,1.24,9.70,0.62,1.71,660.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,12.42,2.55,2.27,22.0,90.0,1.68,1.84,0.66,1.42,2.70,0.86,3.30,315.0,1
85,12.67,0.98,2.24,18.0,99.0,2.20,1.94,0.30,1.46,2.62,1.23,3.16,450.0,1
66,13.11,1.01,1.70,15.0,78.0,2.98,3.18,0.26,2.28,5.30,1.12,3.18,502.0,1
25,13.05,2.05,3.22,25.0,124.0,2.63,2.68,0.47,1.92,3.58,1.13,3.20,830.0,0


In [610]:
# now lets split .. we will keep it as dataframe as we will need it for futher random sampling in bagging 

X=df.drop('target',axis=1)
y=df['target']

df_train,df_test=train_test_split(df,test_size=0.2,random_state=42)

In [611]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [612]:
dt1=DecisionTreeClassifier()
dt1.fit(X_train,y_train)
ypred=dt1.predict(X_test)
accuracy_score(y_test,ypred)
# checking if irt is prooer 

0.8611111111111112

In [613]:
print(df_train.shape)
df_train

(142, 14)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
31,13.58,1.66,2.36,19.1,106.0,2.86,3.19,0.22,1.95,6.90,1.09,2.88,1515.0,0
71,13.86,1.51,2.67,25.0,86.0,2.95,2.86,0.21,1.87,3.38,1.36,3.16,410.0,1
161,13.69,3.26,2.54,20.0,107.0,1.83,0.56,0.50,0.80,5.88,0.96,1.82,680.0,2
19,13.64,3.10,2.56,15.2,116.0,2.70,3.03,0.17,1.66,5.10,0.96,3.36,845.0,0
103,11.82,1.72,1.88,19.5,86.0,2.50,1.64,0.37,1.42,2.06,0.94,2.44,415.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51,13.83,1.65,2.60,17.2,94.0,2.45,2.99,0.22,2.29,5.60,1.24,3.37,1265.0,0
49,13.94,1.73,2.27,17.4,108.0,2.88,3.54,0.32,2.08,8.90,1.12,3.10,1260.0,0
13,14.75,1.73,2.39,11.4,91.0,3.10,3.69,0.43,2.81,5.40,1.25,2.73,1150.0,0
142,13.52,3.17,2.72,23.5,97.0,1.55,0.52,0.50,0.55,4.35,0.89,2.06,520.0,2


In [614]:
df_test.shape

(36, 14)

In [615]:

X_train = df_train.drop("target", axis=1)
X_test = df_test.drop("target", axis=1)

y_train = df_train["target"]
y_test = df_test["target"]

dt = DecisionTreeClassifier()

dt.fit(X_train, y_train)

ypred2 = dt.predict(X_test)

print(accuracy_score(y_test, ypred2))

0.8611111111111112


So decsion tree independent has accuracy of 97% . 

### Bagging

In [616]:
# Bagging - we will see it manully how it works . as core idea we disscued above . 

# we are doing a multi-classification problem above... 

# we have 

df_train 

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
31,13.58,1.66,2.36,19.1,106.0,2.86,3.19,0.22,1.95,6.90,1.09,2.88,1515.0,0
71,13.86,1.51,2.67,25.0,86.0,2.95,2.86,0.21,1.87,3.38,1.36,3.16,410.0,1
161,13.69,3.26,2.54,20.0,107.0,1.83,0.56,0.50,0.80,5.88,0.96,1.82,680.0,2
19,13.64,3.10,2.56,15.2,116.0,2.70,3.03,0.17,1.66,5.10,0.96,3.36,845.0,0
103,11.82,1.72,1.88,19.5,86.0,2.50,1.64,0.37,1.42,2.06,0.94,2.44,415.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51,13.83,1.65,2.60,17.2,94.0,2.45,2.99,0.22,2.29,5.60,1.24,3.37,1265.0,0
49,13.94,1.73,2.27,17.4,108.0,2.88,3.54,0.32,2.08,8.90,1.12,3.10,1260.0,0
13,14.75,1.73,2.39,11.4,91.0,3.10,3.69,0.43,2.81,5.40,1.25,2.73,1150.0,0
142,13.52,3.17,2.72,23.5,97.0,1.55,0.52,0.50,0.55,4.35,0.89,2.06,520.0,2


In [617]:
# now for bagaging we will use 3 base models, 

# create the sample first we have around 142 traingn row which we will provide all for mdodel but by creating variaotn 

df_bagging1=df_train.sample(142,replace=True) # we randomly sampled trainng data , with replacement so it might repeatr as wl

X1=df_bagging1.iloc[:,:-1]
y1=df_bagging1.iloc[:,-1]
dt1=DecisionTreeClassifier()
dt1.fit(X1,y1)

# these are our global testing data we will use 
X_test_global=df_test.drop('target',axis=1)
y_train_global= df_train["target"]
y_test_global = df_test["target"]


ypred1=dt1.predict(X_test_global)
accuracy_score(y_test_global,ypred1)

0.9166666666666666

In [618]:
df_bagging2=df_train.sample(142,replace=True) # we randomly sampled trainng data , with replacement so it might repeatr as wl

X2=df_bagging2.iloc[:,:-1]
y2=df_bagging2.iloc[:,-1]
dt2=DecisionTreeClassifier()
dt2.fit(X2,y2)

# these are our global testing data we will use 
X_test_global=df_test.drop('target',axis=1)
y_test_global = df_test["target"]


ypred2=dt2.predict(X_test_global)
accuracy_score(y_test_global,ypred2)

0.9166666666666666

In [619]:
df_bagging3=df_train.sample(142,replace=True) # we randomly sampled trainng data , with replacement so it might repeatr as wl

X3=df_bagging3.iloc[:,:-1]
y3=df_bagging3.iloc[:,-1]
dt3=DecisionTreeClassifier()
dt3.fit(X3,y3)

# these are our global testing data we will use 
X_test_global=df_test.drop('target',axis=1)
y_test_global = df_test["target"]


ypred3=dt3.predict(X_test_global)
accuracy_score(y_test_global,ypred3)

0.8611111111111112

In [620]:
# so we got diffrent diffrent models above now lets do one thing lets check one individually 

df_test.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
57,13.29,1.97,2.68,16.8,102.0,3.00,3.23,0.31,1.66,6.00,1.07,2.84,1270.0,0
92,12.69,1.53,2.26,20.7,80.0,1.38,1.46,0.58,1.62,3.05,0.96,2.06,495.0,1
23,12.85,1.60,2.52,17.8,95.0,2.48,2.37,0.26,1.46,3.93,1.09,3.63,1015.0,0
65,12.37,1.21,2.56,18.1,98.0,2.42,2.65,0.37,2.08,4.60,1.19,2.30,678.0,1
12,13.75,1.73,2.41,16.0,89.0,2.60,2.76,0.29,1.81,5.60,1.15,2.90,1320.0,0


In [621]:
# lests chck of sn 23. what our base model will predict 
ypred1_single = dt1.predict(
    np.array([12.85, 1.60, 2.52, 17.8, 95.0, 2.48, 2.37,
              0.26, 1.46, 3.93, 1.09, 3.63, 1015.0]).reshape(1, 13)
)

ypred2_single = dt2.predict(
    np.array([12.85, 1.60, 2.52, 17.8, 95.0, 2.48, 2.37,
              0.26, 1.46, 3.93, 1.09, 3.63, 1015.0]).reshape(1, 13)
)

ypred3_single = dt3.predict(
    np.array([12.85, 1.60, 2.52, 17.8, 95.0, 2.48, 2.37,
              0.26, 1.46, 3.93, 1.09, 3.63, 1015.0]).reshape(1, 13)
)

print(ypred1_single)
print(ypred2_single)
print(ypred3_single)

[0]
[0]
[1]


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


though our bagging modesl miss calssify, it should be 0 in real but majoirt count made it class 1 .. its how bagging works .. 

Bagging also have types 



In [627]:
# 1. Pasting - it simply means row sampling without replacement. that it 

# we will take small sample and see, if it actually repeat or not 
# to do pasting simply do .sample(no of row you want) dont add replacemnt parameter 

sample1=df_train.sample(8)
sample2=df_train.sample(8)
sample3=df_train.sample(8)

In [629]:
print(sample1)
print(sample2)
print(sample3)


# so in singel sample pick it wont be repeating 

     alcohol  malic_acid   ash  ...  od280/od315_of_diluted_wines  proline  target
141    13.36        2.56  2.35  ...                          2.47    780.0       2
60     12.33        1.10  2.28  ...                          1.67    680.0       1
87     11.65        1.67  2.62  ...                          3.21    562.0       1
90     12.08        1.83  2.32  ...                          2.27    480.0       1
17     13.83        1.57  2.62  ...                          2.57   1130.0       0
98     12.37        1.07  2.10  ...                          2.77    660.0       1
62     13.67        1.25  1.92  ...                          2.46    630.0       1
33     13.76        1.53  2.70  ...                          3.00   1235.0       0

[8 rows x 14 columns]
     alcohol  malic_acid   ash  ...  od280/od315_of_diluted_wines  proline  target
98     12.37        1.07  2.10  ...                          2.77    660.0       1
10     14.10        2.16  2.30  ...                          3.1

In [ ]:
# Random subspace - it simply means coloum sampling 

# we sample data bsaed on col 
column1=df_train.sample(3,replace=True,axis=1)
column2=df_train.sample(3,replace=True,axis=1)
column3=df_train.sample(3,replace=True,axis=1)

# 3 means we are doing samling form random 3 col only 
# can be with replacement or without too 

In [636]:
print(column1)
print(column2)
print(column3)

      ash  color_intensity  color_intensity
31   2.36             6.90             6.90
71   2.67             3.38             3.38
161  2.54             5.88             5.88
19   2.56             5.10             5.10
103  1.88             2.06             2.06
..    ...              ...              ...
51   2.60             5.60             5.60
49   2.27             8.90             8.90
13   2.39             5.40             5.40
142  2.72             4.35             4.35
18   2.48             8.70             8.70

[142 rows x 3 columns]
     alcohol  alcohol  target
31     13.58    13.58       0
71     13.86    13.86       1
161    13.69    13.69       2
19     13.64    13.64       0
103    11.82    11.82       1
..       ...      ...     ...
51     13.83    13.83       0
49     13.94    13.94       0
13     14.75    14.75       0
142    13.52    13.52       2
18     14.19    14.19       0

[142 rows x 3 columns]
     malic_acid  total_phenols  od280/od315_of_diluted_wines
31 

In [641]:
# Random patchs - both row and col sampling is done 

# we will see with replacement one 

random1=df_train.sample(8,replace=True).sample(3,replace=True,axis=1)

random2=df_train.sample(8,replace=True).sample(3,replace=True,axis=1)

random3=df_train.sample(8,replace=True).sample(3,replace=True,axis=1)


In [642]:
print(random1)
print(random2)
print(random3)

     alcohol   hue  alcohol
107    12.72  0.88    12.72
141    13.36  0.70    13.36
126    12.43  0.69    12.43
55     13.56  0.98    13.56
4      13.24  1.04    13.24
143    13.62  0.91    13.62
74     11.96  0.99    11.96
46     14.38  1.04    14.38
      hue  target   hue
118  0.70       1  0.70
55   0.98       0  0.98
20   1.09       0  1.09
101  1.04       1  1.04
61   0.98       1  0.98
60   1.25       1  1.25
102  0.80       1  0.80
41   0.91       0  0.91
     nonflavanoid_phenols   ash  proanthocyanins
129                  0.42  2.38             1.35
137                  0.63  2.64             1.10
14                   0.29  2.38             2.96
58                   0.19  2.50             2.04
40                   0.34  2.31             2.34
129                  0.42  2.38             1.35
74                   0.13  2.30             1.65
105                  0.66  2.27             1.42
